In [2]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

# =========================
# LOAD DATASET
# =========================
file_path = "merged_final_v3.csv"
df = pd.read_csv(file_path)

print(" Dataset loaded")
print("Shape:", df.shape)

# =========================
# CHECK LABEL COLUMN
# =========================
if "label" not in df.columns:
    raise ValueError(" 'label' column not found in dataset")

# =========================
# SEPARATE LABEL
# =========================
y = df["label"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))

# =========================
# CLEAN FEATURES
# =========================
X = df.drop(columns=["label"])
X = X.select_dtypes(include=[np.number])

print(" Numeric features:", X.shape[1])

# =========================
# HANDLE MISSING VALUES
# =========================
X = X.fillna(X.mean())

# =========================
# NORMALIZATION
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(" Data normalized")

# =========================
# TRAIN MODEL
# =========================
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print(" Training model...")
model.fit(X_scaled, y_encoded)

# =========================
# SHAP VALUES
# =========================
print(" Calculating SHAP values...")

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

# =========================
# FIX SHAP DIMENSIONS
# =========================
# Case 1: list (binary classification)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Case 2: 3D array (samples, features, classes)
elif len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

# Now it should be (samples, features)
print("SHAP shape:", shap_values.shape)

# =========================
# FEATURE IMPORTANCE
# =========================
shap_importance = np.abs(shap_values).mean(axis=0)

# Ensure correct shape
if len(shap_importance) != X.shape[1]:
    raise ValueError(" SHAP importance mismatch with feature count")

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": shap_importance
})

importance_df = importance_df.sort_values(by="importance", ascending=False)

# =========================
# SELECT TOP FEATURES
# =========================
n_features = min(600, X.shape[1])

selected_genes = importance_df["feature"].head(n_features).values

print(f" Selected {len(selected_genes)} genes using SHAP")

# =========================
# FINAL DATASET
# =========================
X_selected = X_scaled[selected_genes]

final_df = pd.DataFrame(X_selected, columns=selected_genes)
final_df["label"] = y.values

print("Final dataset shape:", final_df.shape)

# =========================
# SAVE FILE
# =========================
output_file = "selected_genes_shap.csv"
final_df.to_csv(output_file, index=False)

print(f"DONE: File saved as '{output_file}'")

 Dataset loaded
Shape: (6322, 16482)
Classes: [np.int64(0), np.int64(1)]
 Numeric features: 16479
 Data normalized
 Training model...
 Calculating SHAP values...
SHAP shape: (6322, 16479)
 Selected 600 genes using SHAP
Final dataset shape: (6322, 601)
DONE: File saved as 'selected_genes_shap.csv'
